# Task3：ResNet34固定43轮，全量多随机种子对照

固定原有效版本的全部训练设置，训练种子**42、2024、3407**，每个模型使用全部80张图。每个模型执行43轮，**余弦学习率周期仍是80轮**。不使用局部精修、不做TTA，不重新搜索训练轮数或融合权重。

保存3个单模型提交，以及预先固定的三模型等权坐标平均。没有独立验证集，不根据训练拟合误差、预测一致性或公开榜挑选种子/子集。

原历史单模型公开分数为12.31624；上轮普通五折集成12.37299，精修五折集成16.45402，均为公开榜得分。这里的12.31624**只属于已保存的原历史文件**，不是新seed42模型的成绩。不同GPU/软件环境下，同种子不保证逐位重现。

Notebook已内置历史40行提交及图像指纹作为对照，**只需添加2026task3比赛数据**，不需上轮ZIP或模型权重。首次可能下载约87MB官方ImageNet ResNet34权重。


## 1. 配置与本地保护

在Kaggle导入此Notebook，添加比赛数据，启用GPU和Internet。新输出目录与此前实验分开。只用cuda:0，3个种子顺序运行。本地在导入Torch前停止，保护正在运行的CPU实验。

RUN_SEEDS可先设[42]完成对照，再改回全部3个；已完成模型自动复用。训练轮数与超参数固定在协议中，修改实现必须使用新目录。


In [ ]:
from pathlib import Path
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('请在Kaggle GPU运行。本机不会导入Torch或启动训练。')
DATA_DIR=None
OUTPUT_DIR=Path('/kaggle/working/task3_resnet34_multiseed_full')
RESUME_DIR=None  # 本版完整Output目录，里面直接含protocol.json
WEIGHTS=None    # 可挂载官方resnet34-b627a593.pth，否则自动查找/下载
RUN_SEEDS=[42,2024,3407]
KEEP_LAST_CHECKPOINT=False  # 一模型完整导出后，删除其自身optimizer last.pt，保留最终模型
CREATE_FULL_BACKUP=True


In [ ]:
from __future__ import annotations
from collections import defaultdict
from contextlib import contextmanager
import base64, csv, gc, hashlib, io, json, math, os, random, shutil, statistics, sys, time, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
WEIGHT_NAME='resnet34-b627a593.pth'
WEIGHT_URL='https://download.pytorch.org/models/'+WEIGHT_NAME

def save_json(path, value):
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def find_data(explicit=None):
    if explicit:
        candidates = [Path(explicit)]
    else:
        base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
        candidates = [base / '2026task3/detection', Path.cwd() / '2026task3/detection',
                      Path.cwd() / 'task3/2026task3/detection']
        if Path('/kaggle/input').exists():
            candidates += [p.parent for p in Path('/kaggle/input').rglob('fovea_localization_train_GT.csv')]
    valid = sorted({p.resolve() for p in candidates
                    if (p / 'fovea_localization_train_GT.csv').is_file()
                    and (p / 'sample_submission.csv').is_file()
                    and (p / 'train').is_dir() and (p / 'test').is_dir()})
    if len(valid) != 1:
        raise ValueError(f'Expected one dataset; set data_dir explicitly. Found: {valid}')
    return valid[0]


def load_manifest(data):
    labels = pd.read_csv(data / 'fovea_localization_train_GT.csv')
    if list(labels.columns) != ['data', 'Fovea_X', 'Fovea_Y'] or labels.isna().any().any():
        raise ValueError('Invalid training CSV columns or missing values.')
    if labels['data'].duplicated().any():
        raise ValueError('Duplicate training IDs.')
    lookup = labels.set_index('data')
    records = []
    for subset in ['train', 'test']:
        files = sorted(p for p in (data / subset).iterdir()
                       if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        for p in files:
            image_id = int(p.stem)
            with Image.open(p) as im:
                im = im.convert('RGB')
                im.load()
                w, h = im.size
                # Hash decoded pixels, not JPEG metadata. Size participates in the hash.
                digest = hashlib.sha256(f'{w},{h}:'.encode() + im.tobytes()).hexdigest()
            record = dict(image_id=image_id, subset=subset, filename=p.name,
                          width=w, height=h, pixel_sha256=digest)
            if subset == 'train':
                if image_id not in lookup.index:
                    raise ValueError(f'Missing label: {p.name}')
                x, y = lookup.loc[image_id, ['Fovea_X', 'Fovea_Y']].astype(float)
                if not np.isfinite([x, y]).all() or not (0 <= x < w and 0 <= y < h):
                    raise ValueError(f'Coordinate outside image: {p.name}')
                record.update(x=float(x), y=float(y), nx=float(x / w), ny=float(y / h))
                xml_path = data / 'train_location' / f'{image_id:04}.xml'
                if not xml_path.is_file():
                    raise ValueError(f'Missing XML: {xml_path}')
                xml = ET.parse(xml_path).getroot()
                if (int(xml.findtext('size/width')), int(xml.findtext('size/height'))) != (w, h):
                    raise ValueError(f'XML size mismatch: {p.name}')
                objects = [v for v in xml.findall('object') if v.findtext('name').lower() == 'fovea']
                if len(objects) != 1:
                    raise ValueError(f'Expected one Fovea box: {xml_path}')
                box = objects[0].find('bndbox')
                xmin, ymin, xmax, ymax = [float(box.findtext(k)) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
                if not (xmin <= x <= xmax and ymin <= y <= ymax):
                    raise ValueError(f'CSV coordinate outside XML box: {p.name}')
                record.update(xml_center_dx=float(x - (xmin + xmax) / 2),
                              xml_center_dy=float(y - (ymin + ymax) / 2))
            records.append(record)
    manifest = pd.DataFrame(records)
    if set(manifest.loc[manifest.subset == 'train', 'image_id']) != set(labels['data']):
        raise ValueError('Training images and CSV IDs differ.')
    if manifest.image_id.duplicated().any():
        raise ValueError('Image IDs overlap between train/test.')
    cross = manifest.groupby('pixel_sha256').subset.nunique()
    if (cross > 1).any():
        raise ValueError('Identical pixels occur in both train and test; inspect before continuing.')
    return manifest


def assign_folds(train, seed=42, folds=5):
    """Balance exact-duplicate groups across image resolutions; no test labels used."""
    if folds < 2:
        raise ValueError('Need at least two folds.')
    train = train.copy()
    groups = list(train.groupby('pixel_sha256', sort=True))
    rng = np.random.default_rng(seed)
    buckets = defaultdict(list)
    for digest, group in groups:
        row = group.iloc[0]
        buckets[(int(row.width), int(row.height))].append((digest, len(group)))
    total = np.zeros(folds, dtype=int)
    mapping = {}
    for key in sorted(buckets):
        entries = buckets[key]
        rng.shuffle(entries)
        entries.sort(key=lambda pair: -pair[1])
        sizes = np.zeros(folds, dtype=int)
        for digest, count in entries:
            tie = rng.random(folds)
            fold = min(range(folds), key=lambda f: (sizes[f], total[f], tie[f]))
            mapping[digest] = fold
            sizes[fold] += count
            total[fold] += count
    train['fold'] = train.pixel_sha256.map(mapping)
    if train.groupby('pixel_sha256').fold.nunique().max() != 1:
        raise AssertionError('Duplicate group split across folds.')
    return train


def error_metrics(predicted, truth, sizes):
    delta = (np.asarray(predicted, dtype=np.float64) - np.asarray(truth, dtype=np.float64)) * sizes
    distance = np.linalg.norm(delta, axis=1)
    return dict(n=len(distance), mean_distance_px=float(distance.mean()),
                median_distance_px=float(np.median(distance)),
                p90_distance_px=float(np.quantile(distance, .9)),
                mae_x_px=float(np.abs(delta[:, 0]).mean()),
                mae_y_px=float(np.abs(delta[:, 1]).mean()),
                rmse_coordinate_px=float(np.sqrt(np.mean(delta ** 2))))


class FoveaDataset(Dataset):
    def __init__(self, data, records, image_size, augment=False):
        self.records = records.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment
        self.images = []
        self.jitter = transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.12, hue=.02)
        self.tensor = transforms.Compose([transforms.ToTensor(),
                                          transforms.Normalize([.485, .456, .406], [.229, .224, .225])])
        for row in self.records.itertuples():
            with Image.open(data / row.subset / row.filename) as im:
                self.images.append(im.convert('RGB').resize((image_size, image_size), Image.Resampling.BILINEAR))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        im = self.images[index].copy()
        point = (np.array([row.nx, row.ny], dtype=np.float32) if row['subset'] == 'train'
                 else np.array([.5, .5], dtype=np.float32))
        if self.augment:
            if random.random() < .5:
                im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                point[0] = 1 - point[0]
            angle, scale = math.radians(random.uniform(-10, 10)), random.uniform(.92, 1.08)
            a, b = scale * math.cos(angle), -scale * math.sin(angle)
            size, center = self.image_size, self.image_size / 2
            tx, ty = random.uniform(-.035, .035) * size, random.uniform(-.035, .035) * size
            matrix = np.array([[a, b, center + tx - a * center - b * center],
                               [-b, a, center + ty + b * center - a * center], [0, 0, 1]])
            point = ((matrix @ np.array([point[0] * size, point[1] * size, 1]))[:2] / size).astype(np.float32)
            if not (point >= 0).all() or not (point <= 1).all():
                raise ValueError('Augmentation moved label outside image.')
            inverse = np.linalg.inv(matrix)[:2].reshape(-1)
            im = im.transform((size, size), Image.Transform.AFFINE, inverse,
                              resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0))
            im = self.jitter(im)
        return self.tensor(im), torch.from_numpy(point), torch.tensor([row.width, row.height], dtype=torch.float32)


def pretrained_backbone(pretrained=True, explicit=None):
    model = models.resnet18(weights=None)
    if not pretrained:
        return model
    base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
    candidates = [Path(explicit)] if explicit else []
    candidates += [base.parent / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path.cwd().parent / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path.cwd() / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path(torch.hub.get_dir()) / 'checkpoints/resnet18-f37072fd.pth']
    for path in candidates:
        if path.is_file():
            if not hashlib.sha256(path.read_bytes()).hexdigest().startswith('f37072fd'):
                raise ValueError(f'ImageNet checkpoint hash mismatch: {path}')
            model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
            print(f'ImageNet weights: {path}', flush=True)
            return model
    # Official torchvision ImageNet weights only. No eye-dataset or Task2 fine-tuned weights.
    state = models.ResNet18_Weights.IMAGENET1K_V1.get_state_dict(progress=True, check_hash=True)
    model.load_state_dict(state)
    return model


class DecoderBlock(nn.Module):
    def __init__(self, incoming, skip, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming + skip, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True))

    def forward(self, value, skip):
        value = F.interpolate(value, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.layers(torch.cat([value, skip], dim=1))


class FoveaHeatmap(nn.Module):
    def __init__(self, pretrained=True, weights=None):
        super().__init__()
        self.backbone = pretrained_backbone(pretrained, weights)
        self.backbone.fc = nn.Identity()
        self.d3 = DecoderBlock(512, 256, 128)
        self.d2 = DecoderBlock(128, 128, 64)
        self.d1 = DecoderBlock(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)
        self.freeze_bn = pretrained

    def train(self, mode=True):
        super().train(mode)
        if mode and self.freeze_bn:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, image):
        b = self.backbone
        value = b.maxpool(b.relu(b.bn1(b.conv1(image))))
        c1 = b.layer1(value)
        c2 = b.layer2(c1)
        c3 = b.layer3(c2)
        c4 = b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


def heatmap_loss(logits, points, sigma=1.5):
    height, width = logits.shape[-2:]
    yy = torch.arange(height, device=logits.device, dtype=torch.float32) + .5
    xx = torch.arange(width, device=logits.device, dtype=torch.float32) + .5
    distance = (xx[None, None, :] - points[:, 0, None, None] * width) ** 2
    distance = distance + (yy[None, :, None] - points[:, 1, None, None] * height) ** 2
    target = torch.exp(-distance / (2 * sigma ** 2))
    target = target / target.sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
    return -(target.flatten(1) * F.log_softmax(logits.float().flatten(1), dim=1)).sum(1).mean()


def decode_heatmap(logits, radius=3):
    """Local expectation around the peak; cell centers match resize align_corners=False."""
    batch, height, width = logits.shape
    peak = logits.flatten(1).argmax(1)
    py, px = peak // width, peak % width
    yy = torch.arange(height, device=logits.device)[None, :, None]
    xx = torch.arange(width, device=logits.device)[None, None, :]
    mask = (yy - py[:, None, None]).abs() <= radius
    mask = mask & ((xx - px[:, None, None]).abs() <= radius)
    weights = logits.float().masked_fill(~mask, -torch.inf).flatten(1).softmax(1).view(batch, height, width)
    x = (weights * ((xx + .5) / width)).sum((-2, -1))
    y = (weights * ((yy + .5) / height)).sum((-2, -1))
    return torch.stack([x, y], dim=1)


@torch.inference_mode()
def predict(model, loader, device, flip_tta=False):
    model.eval()
    predictions = []
    for images, _, _ in loader:
        images = images.to(device)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            points = decode_heatmap(model(images))
            if flip_tta:
                other = decode_heatmap(model(images.flip(-1)))
                other[:, 0] = 1 - other[:, 0]
                points = (points + other) / 2
        predictions.append(points.cpu().numpy())
    return np.concatenate(predictions)


@contextmanager
def training_lock(out):
    handle = (out / 'training.lock').open('a+b')
    if handle.tell() == 0:
        handle.write(b'0')
        handle.flush()
    handle.seek(0)
    try:
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        yield
    finally:
        handle.close()


def train_model(data, training, validation, out, config, device, model_builder=None, dataset_builder=None):
    out.mkdir(parents=True, exist_ok=True)
    with training_lock(out):
        # Atomic checkpoint replacement temporarily keeps both the old and new files.
        if shutil.disk_usage(out).free < 600 * 1024 ** 2:
            raise RuntimeError(f'Need at least 600 MiB free for atomic checkpoints; choose another output directory: {out}')
        config_path = out / 'config.json'
        if config_path.exists() and json.loads(config_path.read_text(encoding='utf-8')) != config:
            raise ValueError(f'Configuration changed. Use a new output directory: {out}')
        save_json(config_path, config)
        seed_everything(config['seed'])
        dataset_builder = dataset_builder or FoveaDataset
        train_set = dataset_builder(data, training, config['image_size'], augment=True)
        loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True, num_workers=0)
        valid_loader = (DataLoader(dataset_builder(data, validation, config['image_size']),
                                  batch_size=config['batch_size'], shuffle=False, num_workers=0)
                        if validation is not None else None)
        model = (model_builder(config) if model_builder else FoveaHeatmap(config['pretrained'], config['weights'])).to(device)
        backbone_ids = {id(v) for v in model.backbone.parameters()}
        decoder_params = [v for v in model.parameters() if id(v) not in backbone_ids]
        optimizer = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': config['lr_backbone']},
                                       {'params': decoder_params, 'lr': config['lr_decoder']}],
                                      weight_decay=config['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.get('schedule_epochs', config['epochs']),
                                                               eta_min=config['lr_backbone'] * .05)
        scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
        start, best, best_epoch, history = 0, float('inf'), 0, []
        last_path, best_path = out / 'last.pt', out / 'best.pt'
        if last_path.exists():
            checkpoint = torch.load(last_path, map_location='cpu', weights_only=False)
            model.load_state_dict(checkpoint['model'])
            optimizer.load_state_dict(checkpoint['optimizer'])
            scheduler.load_state_dict(checkpoint['scheduler'])
            scaler.load_state_dict(checkpoint['scaler'])
            start, best, best_epoch, history = (checkpoint[k] for k in ['epoch', 'best', 'best_epoch', 'history'])
            random.setstate(checkpoint['python_rng'])
            np.random.set_state(checkpoint['numpy_rng'])
            torch.set_rng_state(checkpoint['torch_rng'])
            if device.type == 'cuda':
                torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
            print(f'Resume {out.name}: epoch {start}', flush=True)
        for epoch in range(start, config['epochs']):
            begin = time.perf_counter()
            model.train()
            total, seen = 0., 0
            for images, points, _ in loader:
                images, points = images.to(device), points.to(device)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
                    loss = heatmap_loss(model(images), points, config['sigma'])
                if not torch.isfinite(loss):
                    raise RuntimeError('Non-finite training loss.')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
                scaler.step(optimizer)
                scaler.update()
                total += float(loss.detach()) * len(images)
                seen += len(images)
            row = dict(epoch=epoch + 1, train_heatmap_ce=total / seen, lr_backbone=optimizer.param_groups[0]['lr'])
            if valid_loader is not None:
                prediction = predict(model, valid_loader, device)
                row.update(error_metrics(prediction, validation[['nx', 'ny']].to_numpy(),
                                         validation[['width', 'height']].to_numpy()))
                score = row['mean_distance_px']
            else:
                score = 0.
            if score < best or valid_loader is None:
                best, best_epoch = score, epoch + 1
                torch.save(dict(model=model.state_dict(), epoch=best_epoch, config=config), out / 'best.tmp')
                (out / 'best.tmp').replace(best_path)
            scheduler.step()
            row['seconds'] = time.perf_counter() - begin
            history.append(row)
            checkpoint = dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                              scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch + 1,
                              best=best, best_epoch=best_epoch, history=history,
                              python_rng=random.getstate(), numpy_rng=np.random.get_state(),
                              torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all())
            torch.save(checkpoint, out / 'last.tmp')
            (out / 'last.tmp').replace(last_path)
            pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
            save_json(out / 'status.json', dict(state='running', epoch=epoch + 1, epochs=config['epochs'],
                                               best_epoch=best_epoch, best_mean_distance_px=best,
                                               pid=os.getpid(), updated_unix=time.time()))
            print(f'{out.name} {epoch+1:03}/{config["epochs"]} ce={row["train_heatmap_ce"]:.4f}'
                  + (f' val={score:.2f}px best={best:.2f}px@{best_epoch}' if valid_loader else '')
                  + f' {row["seconds"]:.1f}s', flush=True)
        checkpoint = torch.load(best_path, map_location='cpu', weights_only=False)
        model.load_state_dict(checkpoint['model'])
        save_json(out / 'status.json', dict(state='completed', best_epoch=best_epoch,
                                           best_mean_distance_px=best if validation is not None else None,
                                           updated_unix=time.time()))
        return model, best_epoch

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':'), allow_nan=False).encode('utf-8')).hexdigest()


def atomic_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    fields = list(fields or (list(rows[0]) if rows else []))
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def read_rows(path):
    with Path(path).open(encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))


def prediction_rows(records, points, fold, labeled):
    if len(records) != len(points):
        raise ValueError('Prediction count differs from image count.')
    rows, seen = [], set()
    for item, point in zip(records, points):
        key, width, height = int(item['image_id']), int(item['width']), int(item['height'])
        if key in seen:
            raise ValueError('Duplicate image ID.')
        seen.add(key)
        nx, ny = float(point[0]), float(point[1])
        if not all(math.isfinite(v) for v in [nx, ny]) or not (0 <= nx < 1 and 0 <= ny < 1):
            raise ValueError(f'Invalid normalized prediction for image {key}.')
        row = dict(image_id=key, fold=int(fold), width=width, height=height,
                   pixel_sha256=str(item['pixel_sha256']), nx=nx, ny=ny,
                   Fovea_X=nx * width, Fovea_Y=ny * height)
        if labeled:
            x, y = float(item['x']), float(item['y'])
            dx, dy = row['Fovea_X'] - x, row['Fovea_Y'] - y
            row.update(gt_x=x, gt_y=y, dx=dx, dy=dy,
                       distance_px=math.hypot(dx, dy), squared_error=dx * dx + dy * dy)
        rows.append(row)
    return rows


def write_submission(template_path, predictions, target):
    lookup = {}
    for row in predictions:
        key = int(row['image_id'])
        for axis, dimension in [('X', 'width'), ('Y', 'height')]:
            value = float(row[f'Fovea_{axis}'])
            if not math.isfinite(value) or not 0 <= value < int(row[dimension]):
                raise ValueError('Invalid original-pixel coordinate.')
            image_id = f'{key}_Fovea_{axis}'
            if image_id in lookup:
                raise ValueError('Repeated submission ID.')
            lookup[image_id] = value
    template = read_rows(template_path)
    ids = [row['ImageID'] for row in template]
    if len(ids) != len(set(ids)) or set(ids) != set(lookup):
        raise ValueError('Submission IDs differ from template.')
    rows = [dict(ImageID=key, value=f'{lookup[key]:.6f}') for key in ids]
    atomic_csv(target, rows, ['ImageID', 'value'])
    check = read_rows(target)
    if [r['ImageID'] for r in check] != ids or any(not math.isfinite(float(r['value'])) for r in check):
        raise AssertionError('Submission round trip failed.')


def resolve_weights(output, explicit=None):
    candidates = [Path(explicit)] if explicit else []
    candidates += [Path(torch.hub.get_dir()) / 'checkpoints' / WEIGHT_NAME]
    if '__file__' in globals():
        candidates.append(Path(__file__).resolve().parent.parent / 'task2/.torch/hub/checkpoints' / WEIGHT_NAME)
    if Path('/kaggle/input').exists():
        candidates += sorted(Path('/kaggle/input').rglob(WEIGHT_NAME))
    path = next((p for p in candidates if p.is_file()), None)
    if path is None:
        path = output / '_pretrained' / WEIGHT_NAME
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.is_file():
            print('Downloading official ResNet34 ImageNet weights (about 87 MB).', flush=True)
            torch.hub.download_url_to_file(WEIGHT_URL, str(path), hash_prefix='b627a593', progress=True)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if not digest.startswith('b627a593'):
        raise ValueError(f'ImageNet weight hash mismatch: {path}')
    save_json(output / 'pretrained_source.json', dict(path=str(path), url=WEIGHT_URL, sha256=digest))
    return path


class ResNet34CV(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        self.backbone = models.resnet34(weights=None)
        if weights is not None:
            self.backbone.load_state_dict(torch.load(weights, map_location='cpu', weights_only=True), strict=True)
        self.backbone.fc = nn.Identity()
        self.d3 = DecoderBlock(512, 256, 128)
        self.d2 = DecoderBlock(128, 128, 64)
        self.d1 = DecoderBlock(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)

    def train(self, mode=True):
        super().train(mode)
        if mode:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images):
        b = self.backbone
        x = b.maxpool(b.relu(b.bn1(b.conv1(images))))
        c1 = b.layer1(x)
        c2 = b.layer2(c1)
        c3 = b.layer3(c2)
        c4 = b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


def restore_backup(source, output, protocol):
    if source is None:
        return
    source = Path(source).resolve()
    if source == output.resolve():
        return
    candidate = source / 'protocol.json'
    if not candidate.is_file() or json.loads(candidate.read_text()) != protocol:
        raise ValueError('Backup protocol/data/code differs. Choose the correct prior output folder.')
    if (output / 'protocol.json').exists():
        print('Current output already has progress; resuming it without merging backup files.', flush=True)
        return
    for path in source.rglob('*'):
        if not path.is_file() or path.is_symlink() or path.suffix in {'.tmp', '.zip'}:
            continue
        if not path.resolve().is_relative_to(source):
            raise ValueError('Backup path escapes its source directory.')
        destination = output / path.relative_to(source)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)


# Historical values are a reference only; never assign their score to a new model.
HISTORICAL_REFERENCE_B64='SW1hZ2VJRCx2YWx1ZQ0KODFfRm92ZWFfWCwxNzUxLjQ3ODc3MA0KODFfRm92ZWFfWSwxMTQ2Ljg3MzIzNg0KODJfRm92ZWFfWCwxNjc2Ljc1Mjg1MQ0KODJfRm92ZWFfWSwxMDk5LjI3MjQ5MA0KODNfRm92ZWFfWCwxNDgyLjYxODY1NA0KODNfRm92ZWFfWSwxMDM3LjAwNjk3NA0KODRfRm92ZWFfWCwxMjU4LjkyMjQ5Nw0KODRfRm92ZWFfWSwxMTA0LjY5NTc5Nw0KODVfRm92ZWFfWCwxMjAxLjQ3NTY5MA0KODVfRm92ZWFfWSw5NzcuNDgyNTY3DQo4Nl9Gb3ZlYV9YLDEyNjYuNDIxNTcyDQo4Nl9Gb3ZlYV9ZLDExMTcuMjI3MzE2DQo4N19Gb3ZlYV9YLDE0NTguMjQ5MTU3DQo4N19Gb3ZlYV9ZLDEwMTguMjY2Nzk3DQo4OF9Gb3ZlYV9YLDEyNzMuMzE2MzUyDQo4OF9Gb3ZlYV9ZLDExMTEuODQ4NzEyDQo4OV9Gb3ZlYV9YLDE0NzAuOTE4MTM2DQo4OV9Gb3ZlYV9ZLDEwMzcuMjY0MzQ3DQo5MF9Gb3ZlYV9YLDE2NzQuNjA3Mjc4DQo5MF9Gb3ZlYV9ZLDEwODguMDUwMjQ2DQo5MV9Gb3ZlYV9YLDE0NzcuNDMxNzIwDQo5MV9Gb3ZlYV9ZLDEwNjIuMTUxMTk0DQo5Ml9Gb3ZlYV9YLDE3MDUuOTQwOTI4DQo5Ml9Gb3ZlYV9ZLDExMDIuOTQ5MDIzDQo5M19Gb3ZlYV9YLDE2OTEuNTM5MTM3DQo5M19Gb3ZlYV9ZLDExNjMuMDAyMjUzDQo5NF9Gb3ZlYV9YLDE0NjEuMjY2Nzk5DQo5NF9Gb3ZlYV9ZLDEwMTcuOTg4MzI0DQo5NV9Gb3ZlYV9YLDE2ODUuNzI5MDkzDQo5NV9Gb3ZlYV9ZLDEwODMuOTYyNjc5DQo5Nl9Gb3ZlYV9YLDE3MjQuNTA3NjAzDQo5Nl9Gb3ZlYV9ZLDEwNzAuNzMyNDc0DQo5N19Gb3ZlYV9YLDE1MDguNDUyMjA5DQo5N19Gb3ZlYV9ZLDEyMTYuOTU2MTM5DQo5OF9Gb3ZlYV9YLDE0NjguMDM4MzQ4DQo5OF9Gb3ZlYV9ZLDEwNjIuNTI5NjgzDQo5OV9Gb3ZlYV9YLDE0NTUuNzI3NzM4DQo5OV9Gb3ZlYV9ZLDEwMzMuMDI1NjIyDQoxMDBfRm92ZWFfWCwxNjk0Ljk3MDg3OA0KMTAwX0ZvdmVhX1ksMTA5Ny4yMTk3MDYNCg=='
HISTORICAL_REFERENCE_SHA256='3fc82d73f6ae91f68d407d66f4cc6e3ecb5c88b04d5ffd1b566facff7e9755d9'
HISTORICAL_TEST_PIXELS={'81': {'width': 2992, 'height': 2000, 'pixel_sha256': '16b57b3ca98d68651b4f1ee88e0dc28a5b38e8e8808713ae4dd5909a8705515b'}, '82': {'width': 2992, 'height': 2000, 'pixel_sha256': '7993d1db17e76b2119639786527b48524ae9e1f5ea3366f0836b02bf0e6dc028'}, '83': {'width': 2992, 'height': 2000, 'pixel_sha256': 'b7b3617181556eec42c25f5cfcedd8345360953af0a78cecdc125a06bbd2b9c4'}, '84': {'width': 2992, 'height': 2000, 'pixel_sha256': 'e1538414c3bb6af352a3de2ad584abca8bffc650676bcd3b0b0d1cfcf41614cd'}, '85': {'width': 1956, 'height': 1934, 'pixel_sha256': '287b85b153c5e5ba3bb604af7a3492455363efb16f20ee156923b901f09778f2'}, '86': {'width': 2992, 'height': 2000, 'pixel_sha256': '1106aea82ccbaa6701eb65830f5157f6141287ba277a2017b2e0be6c0218fb0a'}, '87': {'width': 2992, 'height': 2000, 'pixel_sha256': '1aa036a9fb23c9997e67e60e9ff8e606bbe403ed170052109486d5d864c200c7'}, '88': {'width': 2992, 'height': 2000, 'pixel_sha256': 'c1a01775d305e059f191b3552baae61a58b4955ff34c44af595c03ba7fba7db6'}, '89': {'width': 2992, 'height': 2000, 'pixel_sha256': '13c60695c7d5a88bf202acb0ab9106003c96308cbd851ba5345fa9724b242b8f'}, '90': {'width': 2992, 'height': 2000, 'pixel_sha256': 'b8bf734593c0236f37ec22c03693409f4152b12450f03135289f3ab84369a022'}, '91': {'width': 2992, 'height': 2000, 'pixel_sha256': 'd8e6c1b167b272a45d14d3d0b6b19259aae7b904d284be0f07473986e3de61b3'}, '92': {'width': 2992, 'height': 2000, 'pixel_sha256': '873e312bb28c25c277c777feb61db8eef9a86d6a1bcc272bb70c59f80bdffc72'}, '93': {'width': 2992, 'height': 2000, 'pixel_sha256': '371dfe820fe484b679b0e6e2f4dd27e061b324212b242197eeb6bb093c948904'}, '94': {'width': 2992, 'height': 2000, 'pixel_sha256': 'd8f4c11a16653f7254ca72481dcbe391e207472a1581bab94c24970cc2badcca'}, '95': {'width': 2992, 'height': 2000, 'pixel_sha256': '27257dc3e72cb8aafc7c90c2c743c33518f7e45e25be7b1f37febd9b6d1735f0'}, '96': {'width': 2992, 'height': 2000, 'pixel_sha256': 'd79133b308e74745749ed989521d418622fb93af39a8653f844711b9c4823abd'}, '97': {'width': 2992, 'height': 2000, 'pixel_sha256': 'f771013fbd4f644415f11065fa4f8cf24acf878209ae76d44f840f754fd95721'}, '98': {'width': 2992, 'height': 2000, 'pixel_sha256': '1cdc65acbe24d4970fd5b84cf1aecd88db59ab20a44bdd4dceb2c3d9259b19a4'}, '99': {'width': 2992, 'height': 2000, 'pixel_sha256': '45d7ee23943b01a37889f40c74942c406a10482beea57972160072fd902af374'}, '100': {'width': 2992, 'height': 2000, 'pixel_sha256': '82bd60ead3ea05b1465ee51852fd55dd05e7674df4a9a55eb8f47eaca7bbc61b'}}
HISTORICAL_TRAIN_FINGERPRINT='fb247663f8e9d0b2c626aadadaaf6fb256ab68c34859bfc732e16e913f0eb291'
_reference=base64.b64decode(HISTORICAL_REFERENCE_B64,validate=True)
if hashlib.sha256(_reference).hexdigest()!=HISTORICAL_REFERENCE_SHA256:
    raise RuntimeError('历史提交参考校验失败，请重新导入完整Notebook。')
HISTORICAL_REFERENCE_ROWS=list(csv.DictReader(io.StringIO(_reference.decode('utf-8-sig'),newline='')))
del _reference


In [ ]:
from __future__ import annotations
MULTISEED_SOURCE_SHA256={'task3_pipeline.py': '5a27db4b7901d185eec831080a16d783ad001adc22b639474d345fe7b315ec3f', 'task3_kaggle_cv.py': '2227703a35b3955b8ce02801930b0bf496703decece4ad5234a1d1f70a0b12e6', 'task3_kaggle_multiseed.py': '3c9f77456f3f8ca41bbbb9a97d79ba3a6ecc0e58ce58d012540587abdf37016f'}
"""Fixed 43/80-epoch-schedule ResNet34 full-data runs, differing only by seed."""
import csv
import gc
import hashlib
import json
import math
from pathlib import Path
import shutil
import statistics
import sys
import time
import zipfile


import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


MULTISEED_VERSION = 'resnet34-full-multiseed-v1'
PLAN_SEEDS = (42,2024,3407)


def multiseed_sources():
    if 'MULTISEED_SOURCE_SHA256' in globals():
        return dict(MULTISEED_SOURCE_SHA256)
    base = Path(__file__).resolve().parent
    return {name:hashlib.sha256((base/name).read_bytes()).hexdigest()
            for name in ('task3_pipeline.py','task3_kaggle_cv.py','task3_kaggle_multiseed.py')}


def manifest_records(records,labeled):
    result=[]
    for r in records:
        row=dict(image_id=int(r['image_id']),width=int(r['width']),height=int(r['height']),
                 subset=r['subset'],filename=r['filename'],pixel_sha256=r['pixel_sha256'])
        if labeled:
            row.update(x=float(r['x']),y=float(r['y']))
        result.append(row)
    return result


def historical_training_fingerprint(records):
    values=manifest_records(records,True)
    # Source CSV labels have six decimals; ignore parser noise below that scale.
    for row in values:
        row['x'],row['y']=round(row['x'],6),round(row['y'],6)
    return stable_hash(values)


def validate_seed_predictions(rows,records,seed):
    expected={int(r['image_id']):r for r in records}
    result={int(r['image_id']):r for r in rows}
    if len(result)!=len(rows) or set(result)!=set(expected):
        raise ValueError('Seed prediction IDs differ or repeat.')
    for key,row in result.items():
        source=expected[key]
        if int(row['seed'])!=seed or row['pixel_sha256']!=source['pixel_sha256']:
            raise ValueError('Seed or image hash differs.')
        for field in ('width','height'):
            if int(row[field])!=int(source[field]):raise ValueError('Image dimensions differ.')
        for axis,dim in (('X','width'),('Y','height')):
            value=float(row[f'Fovea_{axis}'])
            if not math.isfinite(value) or not 0<=value<int(source[dim]):
                raise ValueError('Prediction outside original image.')
    return result


def average_seed_predictions(predictions,records,seeds=PLAN_SEEDS):
    if set(predictions)!=set(seeds):
        raise ValueError('All three predeclared seeds are required for the ensemble.')
    maps=[validate_seed_predictions(predictions[seed],records,seed) for seed in seeds]
    return [dict(image_id=int(r['image_id']),width=int(r['width']),height=int(r['height']),
                 Fovea_X=statistics.mean(float(m[int(r['image_id'])]['Fovea_X']) for m in maps),
                 Fovea_Y=statistics.mean(float(m[int(r['image_id'])]['Fovea_Y']) for m in maps)) for r in records]


def validate_historical_reference(rows,records,expected_pixels):
    lookup={r['ImageID']:float(r['value']) for r in rows}
    expected={f'{int(r["image_id"])}_Fovea_{axis}' for r in records for axis in ('X','Y')}
    if len(rows)!=len(lookup) or set(lookup)!=expected:
        raise ValueError('Historical reference submission IDs differ.')
    result=[]
    for r in records:
        key=int(r['image_id'])
        prior=expected_pixels.get(str(key))
        if prior is None or (int(prior['width']),int(prior['height']),prior['pixel_sha256']) != (
                int(r['width']),int(r['height']),r['pixel_sha256']):
            raise ValueError('Historical reference test-image fingerprint differs.')
        x,y=lookup[f'{key}_Fovea_X'],lookup[f'{key}_Fovea_Y']
        if not (math.isfinite(x+y) and 0<=x<int(r['width']) and 0<=y<int(r['height'])):
            raise ValueError('Historical reference coordinate outside image.')
        result.append(dict(image_id=key,width=int(r['width']),height=int(r['height']),Fovea_X=x,Fovea_Y=y))
    return result


def full_seed_config(protocol,seed):
    if seed not in PLAN_SEEDS:raise ValueError('Undeclared random seed.')
    # weights is a portable identifier. The model builder receives the resolved
    # runtime path separately, so moving a complete backup does not change config.
    return dict(protocol,seed=seed,folds=5,fold='full',model_name='resnet34',
                weights='resnet34-b627a593.pth',pretrained=True,
                architecture='resnet34_same_decoder_as_legacy_fulltrain',
                normalization='imagenet',selection_metric='fixed_final_epoch_no_validation')


def checkpoint_final(path,config):
    item=torch.load(path,map_location='cpu',weights_only=True)
    if int(item.get('epoch',0))!=43 or item.get('config')!=config:
        raise ValueError('Expected this seed\'s fixed final epoch43 checkpoint.')
    del item


def seed_checkpoint_summary(folder,config,verify_model=True):
    metadata=json.loads((folder/'seed_complete.json').read_text())
    signature=stable_hash(config)
    if metadata['signature']!=signature or metadata['epoch']!=43:
        raise ValueError('Completed seed configuration differs.')
    model_path=folder/'checkpoints/best.pt'
    if not model_path.is_file() or hashlib.sha256(model_path.read_bytes()).hexdigest()!=metadata['checkpoint_sha256']:
        raise ValueError('Completed seed model missing/corrupt; mount its full output.')
    if verify_model:checkpoint_final(model_path,config)
    return metadata


def write_seed_reports(data,train,test,reference,output,protocol,verify_models=True):
    test_records=test.to_dict('records')
    predictions,summary={},[]
    for seed in PLAN_SEEDS:
        folder=output/f'seed_{seed}'
        if not (folder/'seed_complete.json').exists():continue
        config=full_seed_config(protocol,seed)
        done=seed_checkpoint_summary(folder,config,verify_models)
        rows=read_rows(folder/'test_predictions.csv')
        validate_seed_predictions(rows,test_records,seed)
        predictions[seed]=rows
        history=read_rows(folder/'checkpoints/history.csv')
        if [int(r['epoch']) for r in history]!=list(range(1,44)):
            raise ValueError('Seed history does not contain all43 epochs.')
        # These are training-fit scores, never candidate selection scores.
        fit=json.loads((folder/'train_fit_metrics.json').read_text())
        summary.append(dict(seed=seed,epoch=43,training_images=80,
            train_fit_rmse_px=float(fit['rmse_coordinate_px']),train_fit_mean_distance_px=float(fit['mean_distance_px']),
            final_heatmap_ce=float(history[-1]['train_heatmap_ce']),
            evaluation='in_sample_training_fit_only',public_score=''))
        write_submission(data/'sample_submission.csv',rows,output/f'submission_resnet34_seed{seed}.csv')
    if summary:atomic_csv(output/'train_fit_summary.csv',summary)
    complete=set(predictions)==set(PLAN_SEEDS)
    plan=dict(complete=complete,completed_seeds=sorted(predictions),planned_seeds=list(PLAN_SEEDS),
        training_images_per_model=80,final_epoch=43,cosine_schedule_epochs=80,validation_used=False,
        ensemble_rule='Equal coordinate mean of exactly the three newly trained seeds; no subsets or weight tuning.',
        selection='No best seed is chosen from training fit or prediction agreement.',
        historical_reference_score=12.31624,historical_reference_only=True,
        retrained_seed42_public_score=None,agent_submitted_to_competition=False,
        limitation='No independent validation in this full-data experiment. Same seed on another platform need not reproduce the old predictions or score.')
    if not complete:
        save_json(output/'selection.json',plan)
        return False
    averaged=average_seed_predictions(predictions,test_records)
    atomic_csv(output/'test_multiseed_equal.csv',averaged)
    write_submission(data/'sample_submission.csv',averaged,output/'submission_resnet34_multiseed_equal.csv')
    reference_by_id={int(r['image_id']):r for r in reference}
    mappings={s:validate_seed_predictions(predictions[s],test_records,s) for s in PLAN_SEEDS}
    diagnostics=[]
    for row in averaged:
        key=int(row['image_id'])
        x,y=float(row['Fovea_X']),float(row['Fovea_Y'])
        old=reference_by_id[key]
        control=mappings[42][key]
        rms=math.sqrt(statistics.mean((float(mappings[s][key]['Fovea_X'])-x)**2+
                                     (float(mappings[s][key]['Fovea_Y'])-y)**2 for s in PLAN_SEEDS))
        diagnostics.append(dict(image_id=key,Fovea_X=x,Fovea_Y=y,seed_disagreement_rms_px=rms,
            ensemble_shift_vs_historical_px=math.hypot(x-float(old['Fovea_X']),y-float(old['Fovea_Y'])),
            new_seed42_shift_vs_historical_px=math.hypot(float(control['Fovea_X'])-float(old['Fovea_X']),
                                                       float(control['Fovea_Y'])-float(old['Fovea_Y']))))
    atomic_csv(output/'test_prediction_diagnostics.csv',diagnostics)
    plan.update(candidate='submission_resnet34_multiseed_equal.csv',
        historical_seed42_vs_new_seed42_mean_shift_px=statistics.mean(r['new_seed42_shift_vs_historical_px'] for r in diagnostics),
        historical_seed42_vs_new_seed42_max_shift_px=max(r['new_seed42_shift_vs_historical_px'] for r in diagnostics),
        candidate_public_score=None)
    save_json(output/'selection.json',plan)
    print('All three seeds complete. Candidate:',output/plan['candidate'],flush=True)
    return True


def multiseed_archive(output,full=False):
    target=(output.parent/'task3_multiseed_full_backup.zip' if full else output/'task3_multiseed_results_light.zip')
    temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
        for path in sorted(output.rglob('*')):
            if not path.is_file() or path.is_symlink() or path.suffix in ('.tmp','.zip'):continue
            if full or path.suffix in ('.csv','.json','.png','.md'):
                archive.write(path,path.relative_to(output))
    temporary.replace(target)
    return target


def prepare_multiseed(reference_rows,reference_sha256,reference_pixels,reference_train_fingerprint,data_dir=None,
                      output_dir='/kaggle/working/task3_resnet34_multiseed_full',resume_dir=None):
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Run this Notebook on Kaggle; no local training was started.')
    if not torch.cuda.is_available():
        raise RuntimeError('Enable Kaggle GPU; CPU training is disabled.')
    data,output=find_data(data_dir),Path(output_dir).resolve()
    manifest=load_manifest(data)
    train=assign_folds(manifest[manifest.subset=='train'],42,5)
    test=manifest[manifest.subset=='test'].copy()
    if len(train)!=80 or len(test)!=20 or train.pixel_sha256.nunique()!=79:
        raise ValueError('Unexpected training/test counts or duplicate groups.')
    if historical_training_fingerprint(train.to_dict('records'))!=reference_train_fingerprint:
        raise ValueError('Training pixels/labels/order differ from the original full-data reference.')
    reference=validate_historical_reference(reference_rows,test.to_dict('records'),reference_pixels)
    protocol=dict(version=MULTISEED_VERSION,seeds=list(PLAN_SEEDS),epochs=43,schedule_epochs=80,
        image_size=320,batch_size=8,lr_backbone=1e-4,lr_decoder=1e-3,weight_decay=1e-4,sigma=1.5,
        split_seed=42,training_mode='all80_no_validation_fixed_epoch',flip_tta=False,
        dataset_fingerprint=stable_hash(manifest_records(train.to_dict('records'),True)),
        test_fingerprint=stable_hash(manifest_records(test.to_dict('records'),False)),
        historical_submission_sha256=reference_sha256,source_sha256=multiseed_sources(),
        training_implementation='Original task3_pipeline.train_model, including80-epoch cosine schedule and every-epoch resume saves')
    output.mkdir(parents=True,exist_ok=True)
    if shutil.disk_usage(output).free<3*1024**3:
        raise RuntimeError('Need at least3 GiB free for models, optimizer and atomic saves.')
    restore_backup(resume_dir,output,protocol)
    if (output/'protocol.json').exists() and json.loads((output/'protocol.json').read_text())!=protocol:
        raise ValueError('Existing multi-seed protocol differs. Use a new output directory.')
    save_json(output/'protocol.json',protocol)
    train.to_csv(output/'train_manifest.csv',index=False)
    test.to_csv(output/'test_manifest.csv',index=False)
    write_submission(data/'sample_submission.csv',reference,output/'historical_reference_12p31624.csv')
    atomic_csv(output/'historical_reference_predictions.csv',reference)
    save_json(output/'environment.json',dict(torch=torch.__version__,torchvision=__import__('torchvision').__version__,
        numpy=np.__version__,gpu=torch.cuda.get_device_name(0),visible_gpu_count=torch.cuda.device_count(),
        device_policy='cuda:0 sequential seeds; original CPU thread setting4; workers0',
        historical_environment='RTX4060 Laptop, torch2.11.0+cu128, numpy2.4.6; same seed is not a bitwise guarantee.'))
    print('80 images /79 pixel groups; seeds',PLAN_SEEDS,';43 epochs /80 cosine period; no validation/TTA.',flush=True)
    print('Historical score12.31624 belongs only to the old submission, not a new seed42 run.',flush=True)
    return dict(data=data,output=output,train=train,test=test,reference=reference,protocol=protocol)


def multiseed_eta(output,current_seed):
    timing=[]
    pending=0
    for seed in PLAN_SEEDS:
        folder=output/f'seed_{seed}'
        history=read_rows(folder/'checkpoints/history.csv') if (folder/'checkpoints/history.csv').exists() else []
        timing.extend(float(r['seconds']) for r in history[-8:])
        if not (folder/'seed_complete.json').exists():pending+=max(0,43-len(history))
    if len(timing)>=3:
        minutes=statistics.median(timing)*pending/60
        save_json(output/'eta.json',dict(current_seed=current_seed,remaining_training_minutes_estimate=minutes,
            excludes='Preparation, future cold starts, prediction, compression and some optimizer writes.'))
        return minutes
    return None


def run_multiseed(prepared,run_seeds=PLAN_SEEDS,weights=None,keep_last=False,full_backup=True):
    if not Path('/kaggle/working').is_dir() or not torch.cuda.is_available():
        raise RuntimeError('Use Kaggle GPU; local/CPU training is disabled.')
    if not run_seeds or len(set(run_seeds))!=len(run_seeds) or not set(run_seeds)<=set(PLAN_SEEDS):
        raise ValueError('RUN_SEEDS must be a unique subset of42,2024,3407.')
    data,output,train,test,reference,protocol=[prepared[k] for k in ('data','output','train','test','reference','protocol')]
    pending=[s for s in run_seeds if not (output/f'seed_{s}/seed_complete.json').exists()]
    weight_path=resolve_weights(output,weights) if pending else None
    with training_lock(output):
        try:
            for seed in run_seeds:
                folder=output/f'seed_{seed}'
                config=full_seed_config(protocol,seed)
                if (folder/'seed_complete.json').exists():
                    seed_checkpoint_summary(folder,config)
                    print(f'Reuse completed fixed43-epoch seed{seed}.',flush=True)
                    continue
                folder.mkdir(parents=True,exist_ok=True)
                checkpoints=folder/'checkpoints'
                if (checkpoints/'best.pt').exists() and not (checkpoints/'last.pt').exists():
                    raise ValueError('Unfinished seed has a best model but no optimizer resume checkpoint. Restore full output or start a new directory.')
                if (checkpoints/'last.pt').exists():
                    last=torch.load(checkpoints/'last.pt',map_location='cpu',weights_only=False)
                    if len(last['cuda_rng'])!=torch.cuda.device_count():
                        raise ValueError('Resume with the same visible GPU count as the saved session.')
                    del last
                save_json(output/'status.json',dict(state='training',seed=seed,updated_unix=time.time()))
                start=time.perf_counter()
                # The legacy trainer seeds before dataset/model creation, just as
                # the original43-epoch run. The ResNet34 model is structurally identical.
                model,epoch=train_model(data,train,None,checkpoints,config,torch.device('cuda:0'),
                    model_builder=lambda cfg:ResNet34CV(weight_path),dataset_builder=FoveaDataset)
                if epoch!=43:raise RuntimeError('Fixed final43-epoch run incomplete.')
                checkpoint_final(checkpoints/'best.pt',config)
                test_loader=DataLoader(FoveaDataset(data,test,320),batch_size=8,shuffle=False,num_workers=0)
                points=predict(model,test_loader,torch.device('cuda:0'),flip_tta=False)
                rows=prediction_rows(test.to_dict('records'),points,0,False)
                for row in rows:
                    row.pop('fold');row['seed']=seed
                atomic_csv(folder/'test_predictions.csv',rows)
                write_submission(data/'sample_submission.csv',rows,folder/'submission.csv')
                fit_loader=DataLoader(FoveaDataset(data,train,320),batch_size=8,shuffle=False,num_workers=0)
                fit_prediction=predict(model,fit_loader,torch.device('cuda:0'),flip_tta=False)
                fit=error_metrics(fit_prediction,train[['nx','ny']].to_numpy(),train[['width','height']].to_numpy())
                save_json(folder/'train_fit_metrics.json',dict(**fit,
                    evaluation='in_sample_training_fit_not_validation',used_for_selection=False))
                save_json(folder/'seed_complete.json',dict(signature=stable_hash(config),seed=seed,epoch=43,
                    checkpoint_sha256=hashlib.sha256((checkpoints/'best.pt').read_bytes()).hexdigest(),
                    seconds=time.perf_counter()-start,public_score=None,
                    submitted_to_competition=False,validation_used=False))
                if not keep_last and (checkpoints/'last.pt').exists():(checkpoints/'last.pt').unlink()
                del model,test_loader,fit_loader,points,fit_prediction
                gc.collect();torch.cuda.empty_cache()
                eta=multiseed_eta(output,seed)
                if eta is not None:print(f'Approximate remaining training: {eta:.1f}min (excludes preparation/prediction/backup).',flush=True)
                write_seed_reports(data,train,test,reference,output,protocol)
                multiseed_archive(output)
            complete=write_seed_reports(data,train,test,reference,output,protocol)
            save_json(output/'status.json',dict(state='completed' if complete else 'partial_completed',updated_unix=time.time()))
            multiseed_archive(output)
            if complete and full_backup:
                backup=multiseed_archive(output,full=True)
                with zipfile.ZipFile(backup) as archive:
                    if not all(f'seed_{s}/checkpoints/best.pt' in archive.namelist() for s in PLAN_SEEDS):
                        raise RuntimeError('Full backup lacks a final model.')
                print('Full backup:',backup,flush=True)
        except BaseException as error:
            save_json(output/'status.json',dict(state='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',
                error=str(error),updated_unix=time.time()))
            multiseed_archive(output)
            raise
    return output


## 2. 数据和历史参考检查

使用全部80张训练图，保留完全重复的0039/0077及各自标签，不删除训练记录。核对原训练图片、坐标、行序及测试像素指纹。80张/79个像素组、20张测试图必须匹配。

训练函数直接复用原task3_pipeline.train_model：同图像增强、ImageNet标准化、冻结编码器BatchNorm运行统计、320输入、batch8、热图σ1.5、解码半径3、AdamW编码器/头学习率1e-4/1e-3、weight_decay1e-4、梯度裁剪5、AMP。保留80轮CosineAnnealingLR周期，执行前43轮。

模型使用与原ResNet34分支相同的骨干和解码结构；每个种子均从官方ImageNet权重初始化。原训练函数每轮覆盖best.pt：**这里无验证集，它最终就是第43轮模型**，不是按训练误差挑出的“最佳模型”。


In [ ]:
PREPARED=prepare_multiseed(
    HISTORICAL_REFERENCE_ROWS,HISTORICAL_REFERENCE_SHA256,HISTORICAL_TEST_PIXELS,HISTORICAL_TRAIN_FINGERPRINT,
    data_dir=DATA_DIR,output_dir=OUTPUT_DIR,resume_dir=RESUME_DIR)
print('GPU:',torch.cuda.get_device_name(0))
print('Output:',OUTPUT_DIR)


## 3. 三个固定43轮模型

每轮保存含优化器、调度器、AMP与随机数状态的last.pt，断点在同GPU数量及兼容软件环境下恢复。每个模型完成后默认仅删除自己的last.pt；最终43轮权重best.pt、历史、预测均保留。

第一个模型结束后按其已测epoch耗时估算剩余训练时间；估计不含全部加载、推理、保存和压缩开销。结束后计算80张训练图的拟合指标，仅用于诊断，**不是验证分数，不能据此选择种子**。


In [ ]:
RESULT_DIR=run_multiseed(PREPARED,run_seeds=RUN_SEEDS,weights=WEIGHTS,
    keep_last=KEEP_LAST_CHECKPOINT,full_backup=CREATE_FULL_BACKUP)


## 4. 查看对照与候选

只有全部3个种子完成，才导出submission_resnet34_multiseed_equal.csv。单模型各自导出，不自动评选最好种子。历史12.31624预测只用于报告新seed42复现偏移，不加入本轮三模型平均。

test_prediction_diagnostics.csv中种子分歧及相对历史坐标偏移不是真实误差，不能据此逐图挑选预测或优化公开榜。最终表现由实际比赛评分确认；本Notebook不自动提交比赛。


In [ ]:
from IPython.display import display
print((RESULT_DIR/'selection.json').read_text(encoding='utf-8'))
if (RESULT_DIR/'train_fit_summary.csv').exists():
    print('以下仅为训练集拟合指标，不是验证成绩：')
    display(pd.read_csv(RESULT_DIR/'train_fit_summary.csv'))
if (RESULT_DIR/'test_prediction_diagnostics.csv').exists():
    display(pd.read_csv(RESULT_DIR/'test_prediction_diagnostics.csv').sort_values('seed_disagreement_rms_px',ascending=False))
candidate=RESULT_DIR/'submission_resnet34_multiseed_equal.csv'
if candidate.exists():
    display(pd.read_csv(candidate))
    print('Predeclared equal candidate:',candidate)
print('Light ZIP:',RESULT_DIR/'task3_multiseed_results_light.zip')
print('Full backup:',RESULT_DIR.parent/'task3_multiseed_full_backup.zip')


## 5. 保存与分析

完成后下载task3_multiseed_results_light.zip交回分析。完整备份task3_multiseed_full_backup.zip默认自动生成，含3份最终43轮权重及配置、历史和预测；建议下载到本地D盘保存。

新会话续训需挂载本版完整Output解压目录，并将RESUME_DIR设为直接含本版protocol.json的目录；比赛数据仍需添加。轻量ZIP不能恢复训练。使用自己的可信断点，未完成模型的optimizer恢复应保持原GPU数量。

准备阶段仅进行低内存逻辑/结构检查，没有在本机执行模型或本轮正式训练。跨GPU/软件差异可能影响seed42复现；保存原12.31624模型与CSV，等待新公开分数后再评估。
